In [1]:
import pandas as pd
import glob

# Unificar los ficheros de datos

In [2]:
archivos = glob.glob("Datos/Originales/DowJones/*.csv")
dfs = [pd.read_csv(f) for f in archivos]
df_unificado = pd.concat(dfs, ignore_index=True)
df_unificado

,open,high,low,last,close,volume,date,symbol,exchange
0,213.370,213.41,213.300,NaN,213.300,NaN,2018-08-16T21:30:00+0000,AAPL,IEXG
1,213.280,213.41,213.160,NaN,213.300,NaN,2018-08-16T21:00:00+0000,AAPL,IEXG
2,213.430,213.43,212.985,NaN,213.280,NaN,2018-08-16T20:30:00+0000,AAPL,IEXG
3,213.133,213.73,212.985,NaN,213.480,NaN,2018-08-16T20:00:00+0000,AAPL,IEXG
4,213.273,213.30,213.133,NaN,213.133,NaN,2018-08-16T19:30:00+0000,AAPL,IEXG
...,...,...,...,...,...,...,...,...,...
2744413,234.500,235.60,234.170,235.300,234.090,291094.0,2021-06-29T17:07:00+0000,V,IEXG
2744414,234.500,235.60,234.170,235.235,234.090,290571.0,2021-06-29T17:05:00+0000,V,IEXG
2744415,234.500,235.60,234.170,235.270,234.090,290254.0,2021-06-29T17:04:00+0000,V,IEXG
2744416,234.500,235.60,234.170,235.255,234.090,289854.0,2021-06-29T17:03:00+0000,V,IEXG


In [3]:
print(df_unificado.shape)
print(df_unificado["symbol"].nunique(), "acciones distintas")
print(df_unificado.dtypes)
print(df_unificado.head())

(2744418, 9)
26 acciones distintas
open        float64
high        float64
low         float64
last        float64
close       float64
volume      float64
date            str
symbol          str
exchange        str
dtype: object
      open    high      low  last    close  volume                      date  \
0  213.370  213.41  213.300   NaN  213.300     NaN  2018-08-16T21:30:00+0000   
1  213.280  213.41  213.160   NaN  213.300     NaN  2018-08-16T21:00:00+0000   
2  213.430  213.43  212.985   NaN  213.280     NaN  2018-08-16T20:30:00+0000   
3  213.133  213.73  212.985   NaN  213.480     NaN  2018-08-16T20:00:00+0000   
4  213.273  213.30  213.133   NaN  213.133     NaN  2018-08-16T19:30:00+0000   

  symbol exchange  
0   AAPL     IEXG  
1   AAPL     IEXG  
2   AAPL     IEXG  
3   AAPL     IEXG  
4   AAPL     IEXG  


# Limpiar el dataset

In [4]:
# 1. Nulos por columna (total y %)
print(df_unificado.isnull().sum())
print(df_unificado.isnull().mean() * 100)

open             0
high             0
low              0
last        849956
close       611651
volume      847408
date             0
symbol           0
exchange         0
dtype: int64
open         0.000000
high         0.000000
low          0.000000
last        30.970355
close       22.287093
volume      30.877512
date         0.000000
symbol       0.000000
exchange     0.000000
dtype: float64


- last (31% nulos): last es último valor recogido de la acción. Para este reto se usa close como precio de referencia, así que probablemente se pueda descartar last directamente en vez de imputarla. No aporta nada que no tengamos ya en close.
- close (22% nulos): esta si se necesita, porque de aqui se sacan los retornos logarítmicos y la volatilidad. 
- volume (31% nulos): para el reto, el volumen no es estrictamente necesario para la simulación Monte Carlo ni para el payoff, así que probablemente sea una columna secundaria (útil solo para el análisis descriptivo). 

> **Corregido más abajo (sección 1):** la hipótesis sobre `last` era errónea. En el tramo de datos donde existe `last`, `close` es el cierre del **día anterior** y el precio real es `last`. No se descarta `last`: es la base del cierre diario.

In [5]:
# 2. Duplicados
print("Duplicados exactos:", df_unificado.duplicated().sum())
print("Duplicados por symbol+date:", df_unificado.duplicated(subset=["symbol", "date"]).sum())

Duplicados exactos: 0


Duplicados por symbol+date: 0


In [6]:
# 3. Rango de fechas y nº de filas por acción (para ver si todas cubren el mismo periodo)
df_unificado["date"] = pd.to_datetime(df_unificado["date"])
resumen = df_unificado.groupby("symbol")["date"].agg(["min", "max", "count"])
print(resumen)

                             min                       max   count
symbol                                                            
AAPL   2018-06-06 17:30:00+00:00 2021-07-01 00:00:00+00:00   48723
AMGN   2020-05-29 15:35:00+00:00 2021-07-01 00:00:00+00:00   25296
AXP    2020-05-29 15:30:59+00:00 2021-07-01 00:00:00+00:00   95650
BA     2020-05-29 15:35:51+00:00 2021-07-01 00:00:00+00:00  101436
CAT    2018-06-11 17:30:00+00:00 2021-07-01 00:00:00+00:00  122364
CRM    2020-05-29 15:31:30+00:00 2021-07-01 00:00:00+00:00  108823
CSCO   2020-05-29 15:31:26+00:00 2021-07-01 00:00:00+00:00  108585
CVX    2020-05-29 15:31:23+00:00 2021-07-01 00:00:00+00:00  108876
DIS    2020-05-29 15:35:00+00:00 2021-07-01 00:00:00+00:00  109393
DOW    2020-05-29 15:35:56+00:00 2021-07-01 00:00:00+00:00  106437
GS     2018-06-11 17:30:00+00:00 2021-07-01 00:00:00+00:00  133145
HD     2020-05-29 17:55:51+00:00 2021-07-01 00:00:00+00:00  105219
IBM    2020-05-29 17:56:29+00:00 2021-07-01 00:00:00+00:00  10

- 6 acciones (AAPL, CAT, GS, INTC, NKE, V) empiezan en junio 2018
- El resto (20 acciones) empiezan en mayo/junio 2020
- Todas terminan el 1 de julio de 2021

In [7]:
# 4. Valores de exchange (¿todas las acciones vienen de la misma bolsa/fuente?)
print(df_unificado["exchange"].value_counts())

exchange
IEXG    2744418
Name: count, dtype: int64


In [8]:
# Calculamos la diferencia de tiempo entre filas consecutivas, por cada símbolo
df_unificado = df_unificado.sort_values(["symbol", "date"])
df_unificado["diff_tiempo"] = df_unificado.groupby("symbol")["date"].diff()

# ¿Qué diferencias de tiempo hay y con qué frecuencia aparecen?
print(df_unificado["diff_tiempo"].value_counts().head(15))

diff_tiempo
0 days 00:01:00    2128282
0 days 00:02:00     363519
0 days 00:05:00     113350
0 days 00:03:00      28256
0 days 00:10:00      24415
0 days 00:15:00      20581
0 days 00:30:00      12084
0 days 00:04:00       8282
0 days 17:30:00       2649
0 days 04:01:00       1885
0 days 13:30:00       1723
0 days 00:06:00       1709
0 days 00:07:00       1065
0 days 05:02:00       1059
0 days 00:00:59        737
Name: count, dtype: int64


Tenemos datos intradía a granularidad de minuto, con algunos huecos naturales (fin de semana, cierre de sesión, minutos sin operaciones). No es un dataset diario, es mucho más fino de lo que probablemente se necesite para el reto.  

In [9]:
# 5. Validación de precios: ¿hay casos donde high < low, o close fuera de [low, high]?
inconsistencias = df_unificado[
    (df_unificado["high"] < df_unificado["low"]) |
    (df_unificado["close"] > df_unificado["high"]) |
    (df_unificado["close"] < df_unificado["low"])
]
print("Filas con precios inconsistentes:", len(inconsistencias))

Filas con precios inconsistentes: 761432


In [10]:
# 6. Valores negativos o cero donde no deberían
print("Precios <= 0:", (df_unificado[["open","high","low","close"]] <= 0).sum().sum())
print("Volumen negativo:", (df_unificado["volume"] < 0).sum())

Precios <= 0: 0
Volumen negativo: 0


# Preguntas

1. Enriquecimiento con Yahoo Finance -- > ¿Es para traer las 4 acciones que faltan del Dow Jones, para ampliar el histórico más allá de julio 2021, o para sustituir directamente el intradía por datos diarios limpios de Yahoo Finance? (sacar un dato por dia en vez de tener datos cada 30  mins por ejemplo)

2. Nuestros datos son intradía, con frecuencia que varía entre 1, 5 y 30 minutos según el año y la acción. Para la simulación Monte Carlo (5 años, 20 pasos) parece que necesitamos agregar a datos diarios, que hacemos un dato por dia? o trabajar con frecuencia de 30 minutos...

3. La columna last tiene un 31% de nulos y no la vemos necesaria si usamos close como precio de referencia. Se puede eliminar?

4. 6 de las 26 acciones tienen histórico desde 2018 y las otras 20 solo desde 2020. Si dos de nuestras 3 acciones de la cartera están en grupos distintos, ¿recortamos todas al periodo común (mayo 2020 - julio 2021), o es preferible elegir las 3 dentro del mismo grupo de cobertura?"

5. Un 27,7% de las filas tienen el cierre fuera del rango high/low de su barra. Tiene sentido o deberíamos limpiar/filtrar estas filas antes de agregar?

6. La columna volume tiene un 31% de nulos. No es imprescindible para Monte Carlo/payoff, pero si para el análisis descriptivo. La imputamos (ej. con 0, o interpolando) o la dejamos con nulos y simplemente los excluimos de los gráficos que la usen?"

# Respuestas a las preguntas: implementación

## Criterios acordados con los profesores (23/09/2026)

1. **Fuente principal: los CSV.** Siempre que el CSV tenga un dato, manda el CSV.
2. **Yahoo Finance solo para imputar** lo que el CSV no tiene: los días fuera de su cobertura (2016 → mayo/jun 2020 o jun 2018, y jul 2021 → 2025) y los valores sueltos que falten.
3. **Ventana de trabajo: 2016-01-01 a 2025-12-31.**
4. **Criterio de cobertura: desde que la acción empezó a cotizar**, no desde que entró en el índice. Si en 2016 cotizaba fuera del Dow Jones, se coge igual.
5. **Universo: las 30 empresas del Dow Jones en 2025** (decisión del 25/09/2026), con todo su histórico 2016-2025. Las del CSV que ya no están en el índice en 2025 se quedan fuera; las del índice de 2025 que no están en el CSV vienen enteras de Yahoo (sección 7).
6. **Splits:** hay que comprobar si el `Close` de Yahoo con `auto_adjust=False` ya viene ajustado, contrastándolo con el CSV (sección 3).

## Qué cambia respecto a la versión anterior de este notebook

Al comprobar el CSV contra Yahoo han salido varios errores de la limpieza anterior, que se corrigen aquí:

| Antes | Problema | Ahora |
|---|---|---|
| Cierre diario = último `close` del día | En el tramo con `last`, `close` es el cierre del día anterior (sección 1) | Cierre = último `last` de la sesión; `close` solo donde no existe `last` |
| Día = fecha en UTC | Crea 13.702 filas en sábado (viernes tarde en Nueva York) | Día = fecha en hora de Nueva York |
| Volumen diario = suma | El volumen del CSV es acumulado dentro del día: la suma lo multiplica ×5 | Volumen diario = máximo del día |
| `last` descartada | Era la columna con el precio real | Se usa |
| Volumen sin revisar | Desde el 02/03/2021 el CSV da solo el volumen de IEX (~3% del total) | Se imputa desde Yahoo (sección 4b) |
| Yahoo como fuente única | Contradice el criterio 1 | CSV primero, Yahoo imputa |
| Universo = Dow Jones 2018-2021 (con DOW, INTC y WBA) | DOW no cotizaba antes de 2019 y WBA no tiene datos | Universo = Dow Jones 2025 (criterio 5) |

## 1. Qué significan `close`, `last` y `volume` en los CSV

La exploración de arriba daba un 27,7% de filas con `close` fuera de [low, high]. Antes de agregar a diario miramos qué pasa en un día concreto: CRM el 26/08/2020, un día de resultados en el que la acción subió ~26%.

In [11]:
# Pasamos las marcas de tiempo a hora de Nueva York (la bolsa del Dow Jones)
df_unificado["hora_ny"] = df_unificado["date"].dt.tz_convert("America/New_York")
df_unificado["dia_ny"] = df_unificado["hora_ny"].dt.tz_localize(None).dt.normalize()

ejemplo = df_unificado[(df_unificado["symbol"] == "CRM") & (df_unificado["dia_ny"] == "2020-08-26")]
ejemplo[["hora_ny", "open", "high", "low", "last", "close", "volume"]].iloc[::60]

,hora_ny,open,high,low,last,close,volume
410967,2020-08-26 08:52:00-04:00,249.29,249.29,246.78,246.780,216.05,4599.0
410907,2020-08-26 10:32:00-04:00,249.29,276.14,246.78,272.325,216.05,580805.0
410847,2020-08-26 11:47:00-04:00,249.29,277.79,246.78,271.130,216.05,1171892.0
410787,2020-08-26 13:09:00-04:00,249.29,277.79,246.78,274.185,216.05,1401530.0
410727,2020-08-26 14:25:00-04:00,249.29,277.79,246.78,271.675,216.05,1596052.0
410667,2020-08-26 15:42:00-04:00,249.29,277.79,246.78,269.630,216.05,1738652.0


- `close` se queda fijo en 216,05 todo el día mientras la acción cotiza a ~270: **es el cierre del día anterior**, no el de la barra. Por eso salía fuera de [low, high] en el 27,7% de las filas.
- `last` es el precio real de cada minuto.
- `open` es la apertura del día y `high`/`low` son los extremos acumulados del día hasta ese momento.
- `volume` crece a lo largo del día: **es acumulado**.

Este formato solo aplica al tramo en el que hay `last` (a partir de mayo/junio 2020). En el tramo anterior (jun 2018 → 2020, solo AAPL, CAT, GS, INTC, NKE y V, barras de 15 min) no existe `last` y `close` sí es el cierre de cada barra. Lo comprobamos numéricamente en la sección 4 contra Yahoo.

Confirmamos que el volumen es acumulado en todo el dataset, no solo en este ejemplo:

In [12]:
con_volumen = df_unificado[df_unificado["volume"].notna()].sort_values(["symbol", "date"])
variacion = con_volumen.groupby(["symbol", "dia_ny"])["volume"].diff()
print(f"Pares de minutos consecutivos en los que el volumen crece o se mantiene: {(variacion >= 0).mean():.2%}")
print(f"Pares en los que decrece: {(variacion < 0).mean():.2%}")

Pares de minutos consecutivos en los que el volumen crece o se mantiene: 99.62%
Pares en los que decrece: 0.02%


El volumen no decrece prácticamente nunca dentro de un día, así que el volumen diario es el **máximo** (el último acumulado), no la suma.

## 2. Agregar a frecuencia diaria

**Por qué diaria:** TTM = 5 años y 20 pasos ⇒ dt = 0,25 años por paso. El Montecarlo solo necesita un precio de cierre por día para calibrar σ y ρ; trabajar en intradía solo arrastra la granularidad mixta (1, 5, 15, 30 min) sin aportar nada.

**Por qué fecha de Nueva York y no UTC:** la sesión termina a las 16:00 de Nueva York (20:00/21:00 UTC) y hay after-hours hasta las 20:00. Agrupando por fecha UTC, lo que pasa el viernes por la tarde en Nueva York cae en sábado y crea días de mercado que no existen:

In [13]:
dia_utc = df_unificado["date"].dt.tz_localize(None).dt.normalize()
print("Filas que caen en fin de semana agrupando por UTC:", (dia_utc.dt.dayofweek >= 5).sum())
print("Filas que caen en fin de semana agrupando por Nueva York:", (df_unificado["dia_ny"].dt.dayofweek >= 5).sum())

Filas que caen en fin de semana agrupando por UTC: 13702
Filas que caen en fin de semana agrupando por Nueva York: 0


**Regla del cierre diario** (justificada con la comparación contra Yahoo de la sección 4):

1. Si el día tiene `last`: el último `last` **hasta las 16:00 de Nueva York**. Si se toma el último `last` del día entero, en días de resultados publicados tras el cierre (INTC 22/10/2020, CSCO 12/08/2020, IBM 20/07/2020, NKE 24/06/2021…) se cuela el precio after-hours, con errores de hasta el 10% frente al cierre oficial. Cada error así mete dos retornos falsos en la serie (ida y vuelta) e infla σ. Si no hay `last` antes de las 16:00, se usa el último del día.
2. Si el día no tiene ningún `last` (tramo 2018-2020): el último `close` del día. Aquí no se filtra por hora porque en ese tramo las marcas de tiempo vienen desplazadas (la sesión aparece de 13:30 a 20:00 en hora de Nueva York, es decir, las horas UTC etiquetadas como tal): un filtro de sesión quitaría barras reales.

Resto de columnas: `open` = primero del día, `high` = máximo, `low` = mínimo, `volume` = máximo (acumulado). Se descarta `exchange` (siempre IEXG) y se añade `fuente` para saber de dónde sale cada fila.

In [14]:
import numpy as np

minutos_ny = df_unificado["hora_ny"].dt.hour * 60 + df_unificado["hora_ny"].dt.minute
df_unificado["last_sesion"] = df_unificado["last"].where(minutos_ny <= 16 * 60)

df_diario = (
    df_unificado
    .sort_values(["symbol", "date"])
    .groupby(["symbol", "dia_ny"], as_index=False)
    .agg(
        open=("open", "first"),
        high=("high", "max"),
        low=("low", "min"),
        last_sesion=("last_sesion", "last"),   # último last hasta las 16:00 NY
        last_dia=("last", "last"),             # último last del día (respaldo)
        close_barra=("close", "last"),         # solo válido si el día no tiene last
        n_last=("last", "count"),
        volume=("volume", "max"),
    )
    .rename(columns={"dia_ny": "date"})
)

df_diario["close"] = np.where(
    df_diario["n_last"] > 0,
    df_diario["last_sesion"].fillna(df_diario["last_dia"]),
    df_diario["close_barra"],
)
df_diario = df_diario[["symbol", "date", "open", "high", "low", "close", "volume"]]
df_diario["fuente"] = "csv"

print(df_diario.shape)
print("Días sin cierre (ni last ni close):", df_diario["close"].isna().sum())
df_diario.head()

(10177, 8)
Días sin cierre (ni last ni close): 94


,symbol,date,open,high,low,close,volume,fuente
0,AAPL,2018-06-06,193.630,194.000,191.950,194.00,NaN,csv
1,AAPL,2018-06-07,194.155,194.160,192.385,193.50,NaN,csv
2,AAPL,2018-06-08,190.980,191.995,189.805,191.51,NaN,csv
3,AAPL,2018-06-11,191.345,191.970,190.230,191.16,NaN,csv
4,AAPL,2018-06-12,191.385,192.600,191.175,192.32,NaN,csv


In [15]:
# ¿Qué días se quedan sin cierre?
print(df_diario.loc[df_diario["close"].isna(), "date"].dt.date.value_counts())

date
2021-03-11    26
2021-03-12    26
2021-03-30    26
2021-03-31    16
Name: count, dtype: int64


Son cuatro días de marzo de 2021 en los que el CSV solo trae `open`/`high`/`low` y tanto `last` como `close` vienen vacíos. No se pueden reconstruir con el propio CSV, así que **se imputan enteros desde Yahoo** en la sección 6 (criterio 2). Se imputa la fila completa y no solo el cierre, para no mezclar el OHLC de una fuente con el cierre de la otra dentro de la misma vela.

## 3. Descarga de Yahoo Finance (2016-2025) y comprobación de splits

Se descargan las 30 acciones del Dow Jones **de 2025** (criterio 5) en la ventana completa y con `auto_adjust=False`. Guardamos también los splits que Yahoo registra en la ventana, para comprobar si su `Close` ya viene ajustado.

`FECHA_FIN` es `2026-01-01` porque en yfinance el extremo `end` es exclusivo: así el último día incluido es el 31/12/2025.

In [16]:
import yfinance as yf

FECHA_INICIO = "2016-01-01"
FECHA_FIN = "2026-01-01"   # exclusivo en yfinance -> último día 2025-12-31

# Composición del Dow Jones 30 en 2025 (criterio 5). Frente a la de 2018-2021 del CSV:
# NVDA sustituye a INTC y SHW a DOW (08/11/2024); AMZN sustituye a WBA (26/02/2024)
dow30 = [
    "AAPL", "AMGN", "AMZN", "AXP", "BA", "CAT", "CRM", "CSCO", "CVX", "DIS",
    "GS", "HD", "HON", "IBM", "JNJ", "JPM", "KO", "MCD", "MMM", "MRK",
    "MSFT", "NKE", "NVDA", "PG", "SHW", "TRV", "UNH", "V", "VZ", "WMT",
]
simbolos_csv = set(df_diario["symbol"].unique())
print(f"{len(simbolos_csv & set(dow30))} de {len(dow30)} acciones del Dow Jones 2025 en el CSV")
print("En el Dow Jones 2025 pero no en el CSV (vienen enteras de Yahoo):", sorted(set(dow30) - simbolos_csv))
print("En el CSV pero fuera del Dow Jones 2025 (se descartan):", sorted(simbolos_csv - set(dow30)))

# El CSV se queda solo con las acciones del universo 2025
df_diario = df_diario[df_diario["symbol"].isin(dow30)].reset_index(drop=True)

partes, splits, fallidos = [], {}, []
for ticker in dow30:
    try:
        activo = yf.Ticker(ticker)
        hist = activo.history(start=FECHA_INICIO, end=FECHA_FIN, auto_adjust=False)
        if hist.empty:
            fallidos.append(ticker)
            continue
        hist = hist.reset_index()[["Date", "Open", "High", "Low", "Close", "Volume"]]
        hist.columns = ["date", "open", "high", "low", "close", "volume"]
        hist["date"] = pd.to_datetime(hist["date"]).dt.tz_localize(None).dt.normalize()
        hist["symbol"] = ticker
        partes.append(hist)

        eventos = activo.splits
        eventos.index = eventos.index.tz_localize(None).normalize()
        splits[ticker] = eventos[(eventos.index >= FECHA_INICIO) & (eventos.index < FECHA_FIN)]
    except Exception as error:
        fallidos.append(ticker)
        print(f"{ticker}: falló la descarga -> {error}")

df_yahoo = pd.concat(partes, ignore_index=True)
df_yahoo["fuente"] = "yahoo"
print("Descargadas:", df_yahoo["symbol"].nunique(), "de", len(dow30), "| Fallidas:", fallidos)
df_yahoo.groupby("symbol")["date"].agg(["min", "max", "count"])

24 de 30 acciones del Dow Jones 2025 en el CSV
En el Dow Jones 2025 pero no en el CSV (vienen enteras de Yahoo): ['AMZN', 'HON', 'NVDA', 'SHW', 'VZ', 'WMT']
En el CSV pero fuera del Dow Jones 2025 (se descartan): ['DOW', 'INTC']


C:\Users\Personal\.conda\envs\BDA\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Descargadas: 30 de 30 | Fallidas: []


,min,max,count
symbol,,,
AAPL,2016-01-04,2025-12-31,2514
AMGN,2016-01-04,2025-12-31,2514
AMZN,2016-01-04,2025-12-31,2514
AXP,2016-01-04,2025-12-31,2514
BA,2016-01-04,2025-12-31,2514
CAT,2016-01-04,2025-12-31,2514
CRM,2016-01-04,2025-12-31,2514
CSCO,2016-01-04,2025-12-31,2514
CVX,2016-01-04,2025-12-31,2514


In [17]:
# Eventos que Yahoo registra como split en la ventana
for ticker, eventos in splits.items():
    for fecha, ratio in eventos.items():
        print(f"{ticker:5s} {fecha.date()}  factor {ratio:g}")

AAPL  2020-08-31  factor 4
AMZN  2022-06-06  factor 20
HON   2016-10-03  factor 1.00533
HON   2018-10-01  factor 1.011
HON   2018-10-29  factor 1.032
HON   2025-10-30  factor 1.061
IBM   2021-11-04  factor 1.046
MMM   2024-04-01  factor 1.196
MRK   2021-06-03  factor 1.048
NVDA  2021-07-20  factor 4
NVDA  2024-06-10  factor 10
SHW   2021-04-01  factor 3
WMT   2024-02-26  factor 3


Aparecen dos tipos de evento:
- **Splits reales:** AAPL 4:1 (31/08/2020), SHW 3:1 (01/04/2021), NVDA 4:1 (20/07/2021) y 10:1 (10/06/2024), AMZN 20:1 (06/06/2022) y WMT 3:1 (26/02/2024). Solo el de AAPL cae en una acción del CSV; los demás son de acciones que vienen enteras de Yahoo.
- **Spin-offs que Yahoo trata como split** (factor cercano a 1): MRK 1,048 (Organon, 06/2021), IBM 1,046 (Kyndryl, 11/2021), MMM 1,196 (Solventum, 04/2024) y HON 1,061 (Solstice, 10/2025). Yahoo reduce el precio anterior en la proporción del valor escindido para que la serie no dé un salto el día del spin-off.

**Comprobación: ¿el `Close` de Yahoo con `auto_adjust=False` ya está ajustado?** Si lo está, en cada fecha el cociente `close_csv / close_yahoo` debe ser igual al producto de los factores de todos los eventos **posteriores** a esa fecha: 4 para AAPL antes del 31/08/2020, ~1,046 para IBM en todo el CSV (el spin-off es de nov 2021, después del CSV), etc. Si no lo está, el cociente debe ser ~1 en todas partes.

In [18]:
def factor_ajuste(ticker, fecha):
    """Producto de los factores de split/spin-off posteriores a la fecha (1 si no hay)."""
    eventos = splits.get(ticker)
    if eventos is None or eventos.empty:
        return 1.0
    return float(np.prod(eventos[eventos.index > fecha].values)) if (eventos.index > fecha).any() else 1.0

comparacion = df_diario.merge(
    df_yahoo[["symbol", "date", "close", "volume"]].rename(columns={"close": "close_yahoo", "volume": "volume_yahoo"}),
    on=["symbol", "date"], how="inner",
).dropna(subset=["close"])
comparacion["factor"] = [factor_ajuste(t, f) for t, f in zip(comparacion["symbol"], comparacion["date"])]
comparacion["cociente"] = comparacion["close"] / comparacion["close_yahoo"]

tabla_splits = (
    comparacion.groupby(["symbol", "factor"])
    .agg(cociente_mediano=("cociente", "median"), dias=("cociente", "size"))
    .reset_index()
)
tabla_splits[(tabla_splits["factor"] != 1) | (tabla_splits["symbol"].isin(["AAPL", "MRK"]))].round(4)

,symbol,factor,cociente_mediano,dias
0,AAPL,1.000,1.0001,206
1,AAPL,4.000,4.0000,562
12,IBM,1.046,1.0460,268
17,MMM,1.196,1.1962,267
18,MRK,1.000,1.0004,20
19,MRK,1.048,1.0480,248


**Resultado:** el cociente mediano coincide con el factor en todos los casos (AAPL 4,000 antes del split y 1,000 después; IBM ~1,046; MMM ~1,196; MRK 1,000 antes del spin-off y ~1,048 después; el resto ~1). Es decir, **el `Close` de Yahoo con `auto_adjust=False` ya viene ajustado por splits y spin-offs** (solo quedan fuera los dividendos, que van en `Adj Close`). El CSV, en cambio, trae los precios tal como cotizaron.

**Decisión: ajustar el CSV al criterio de Yahoo** (dividir precios entre el factor y multiplicar el volumen), no al revés:
- La mayor parte de la ventana 2016-2025 va a venir de Yahoo, así que la serie combinada tiene que estar en su escala. Si no, cada frontera CSV↔Yahoo mete un escalón falso (en AAPL, un −75% que no existe).
- Aunque el CSV fuera la única fuente, el split de AAPL cae dentro del propio CSV (31/08/2020): sin ajustar, el retorno logarítmico de ese día sería ln(1/4) ≈ −1,39, que por sí solo dispara σ.
- Ajustar no cambia ningún retorno real: dividir toda la serie anterior por una constante solo mueve el nivel, no las variaciones.

Hacemos lo mismo con el volumen si Yahoo también lo ajusta (se comprueba abajo).

In [19]:
con_vol = comparacion.dropna(subset=["volume"]).copy()
con_vol["cociente_vol"] = con_vol["volume"] / con_vol["volume_yahoo"]
con_vol.groupby(["symbol", "factor"])["cociente_vol"].median().loc[["AAPL", "IBM", "MMM", "MRK"]].round(3)

symbol  factor
AAPL    1.000     0.985
        4.000     0.246
IBM     1.046     0.956
MMM     1.196     0.836
MRK     1.000     0.035
        1.048     0.954
Name: cociente_vol, dtype: float64

El cociente de volumen es el inverso del factor en todos los eventos: AAPL 0,25 = 1/4, IBM ~0,956 = 1/1,046, MMM ~0,836 = 1/1,196 y MRK ~0,954 = 1/1,048. Es decir, **Yahoo ajusta también el volumen con el mismo factor**, tanto en splits como en spin-offs. Para ser coherentes, el volumen del CSV se multiplica por el mismo factor por el que se dividen los precios.

(El 0,035 de MRK antes del spin-off no tiene que ver con el ajuste: son 20 días de mayo de 2021, dentro del tramo en que el volumen del CSV cambia de medida. Se trata en la sección 4.)

In [20]:
df_diario["factor"] = [factor_ajuste(t, f) for t, f in zip(df_diario["symbol"], df_diario["date"])]
for col in ["open", "high", "low", "close"]:
    df_diario[col] = df_diario[col] / df_diario["factor"]
df_diario["volume"] = df_diario["volume"] * df_diario["factor"]

print("Filas del CSV ajustadas:", (df_diario["factor"] != 1).sum())
print(df_diario.loc[df_diario["factor"] != 1].groupby("symbol")["factor"].agg(["min", "max", "count"]))
df_diario = df_diario.drop(columns=["factor"])

Filas del CSV ajustadas: 1374
          min    max  count
symbol                     
AAPL    4.000  4.000    581
IBM     1.046  1.046    271
MMM     1.196  1.196    271
MRK     1.048  1.048    251


## 4. Validación del cierre del CSV contra Yahoo

Con el CSV ya en la misma escala, comparamos el cierre diario de ambas fuentes en los días que comparten. Es la prueba de que la regla del cierre (sección 2) es correcta. No se usa para sustituir el CSV: el criterio es que el CSV manda.

In [21]:
validacion = df_diario.merge(
    df_yahoo[["symbol", "date", "close"]].rename(columns={"close": "close_yahoo"}),
    on=["symbol", "date"], how="inner",
).dropna(subset=["close"])
validacion["error_pct"] = (validacion["close"] / validacion["close_yahoo"] - 1).abs() * 100

print(f"Días comparados: {len(validacion)}")
print(f"Error medio: {validacion['error_pct'].mean():.3f}%  | mediana: {validacion['error_pct'].median():.3f}%  "
      f"| percentil 99: {validacion['error_pct'].quantile(0.99):.3f}%")
print(f"Días con error > 1%: {(validacion['error_pct'] > 1).sum()}  | > 2%: {(validacion['error_pct'] > 2).sum()}  "
      f"| > 5%: {(validacion['error_pct'] > 5).sum()}")
validacion.sort_values("error_pct").tail(8)[["symbol", "date", "close", "close_yahoo", "error_pct"]]

Días comparados: 8932
Error medio: 0.079%  | mediana: 0.039%  | percentil 99: 0.585%
Días con error > 1%: 40  | > 2%: 6  | > 5%: 0


,symbol,date,close,close_yahoo,error_pct
5856,MMM,2020-05-29,128.315217,130.802673,1.901686
6127,MRK,2020-05-29,75.524809,77.022903,1.944998
2839,CSCO,2021-03-26,51.480000,52.570000,2.073425
7305,NKE,2020-12-18,140.170000,137.279999,2.105187
2286,CAT,2021-03-17,228.280000,233.630005,2.289948
772,AMGN,2020-05-29,224.265000,229.699997,2.366128
1328,BA,2020-06-16,202.980000,197.770004,2.634371
2633,CSCO,2020-05-29,46.035000,47.820000,3.732747


Como referencia, estas son las alternativas que probamos para el cierre diario (mismo cálculo):

| Regla | Error medio | Días > 5% |
|---|---|---|
| Último `close` del día, fecha UTC (versión anterior) | 0,92% | 108 |
| Último `close` del día, fecha Nueva York | 0,89% | 101 |
| Último `last` del día entero | 0,04% | 6 (todos after-hours en días de resultados) |
| **Último `last` hasta las 16:00 NY (la elegida)** | **0,08%** | **0** |

Con la regla elegida la media sube un poco frente a usar el día entero (el último minuto antes de las 16:00 no siempre es la subasta de cierre), pero desaparecen todos los errores grandes. Para σ importa más eso: un error del 0,1% apenas cambia la volatilidad, uno del 10% mete dos retornos falsos.

Los días que siguen por encima del 1-2% son pocos y ya no son sistemáticos. **Se mantienen tal cual** porque el criterio es que el CSV manda; quedan anotados en *Pendientes*.

### 4b. Volumen: el CSV cambia de medida en marzo de 2021

Hacemos la misma comparación con el volumen, ya ajustado, agrupando por mes:

In [22]:
vol = df_diario.dropna(subset=["volume"]).merge(
    df_yahoo[["symbol", "date", "volume"]].rename(columns={"volume": "volume_yahoo"}),
    on=["symbol", "date"], how="inner",
)
vol["cociente_vol"] = vol["volume"] / vol["volume_yahoo"]
vol.groupby(vol["date"].dt.to_period("M"))["cociente_vol"].agg(["median", "min"]).round(3)

,median,min
date,,
2020-05,1.000,1.000
2020-06,0.995,0.001
2020-07,1.000,0.011
2020-08,0.999,0.008
2020-09,1.000,0.005
2020-10,0.998,0.011
2020-11,1.000,0.017
2020-12,1.000,0.009
2021-01,1.000,0.023


Hasta febrero de 2021 el volumen del CSV coincide con el de Yahoo (mediana ~1). **Desde el 02/03/2021 pasa a ser ~3%** en todas las acciones del CSV a la vez. El CSV viene de IEX (`exchange = IEXG`), una bolsa con ~3% de la negociación de EE. UU.: a partir de esa fecha el proveedor da el **volumen negociado solo en IEX**, no el consolidado de todas las bolsas. Además, desde junio de 2020 hay filas sueltas que ya venían así (el mínimo de cada mes está entre el 0,1% y el 2%).

**No es un valor faltante, es otra variable.** Si se mantuviera, la serie de volumen caería ×30 de un día para otro sin que el mercado haya cambiado, y cualquier gráfico o estadístico descriptivo del volumen saldría falso. Por eso, **el volumen del CSV que esté por debajo de la mitad del de Yahoo se marca como no válido y se imputa desde Yahoo** (queda indicado en `volume_imputado`). Usamos una regla por fila en vez de una fecha de corte porque así también cubre las filas sueltas anteriores. El umbral del 50% separa con holgura los dos regímenes (~100% frente a ~3%).

No afecta a la valoración: el Montecarlo solo usa `close`. Es una excepción al criterio de que el CSV manda, así que queda en *Pendientes* para confirmarla.

In [23]:
UMBRAL_VOLUMEN = 0.5   # por debajo del 50% del consolidado se considera volumen solo de IEX

df_diario = df_diario.merge(
    df_yahoo[["symbol", "date", "volume"]].rename(columns={"volume": "volume_yahoo"}),
    on=["symbol", "date"], how="left",
)
volumen_iex = (df_diario["volume"] / df_diario["volume_yahoo"]) < UMBRAL_VOLUMEN
print(f"Volúmenes del CSV marcados como solo-IEX: {volumen_iex.sum()} de {df_diario['volume'].notna().sum()}")
print("Primer y último día afectados:", df_diario.loc[volumen_iex, "date"].min().date(), "->", df_diario.loc[volumen_iex, "date"].max().date())
df_diario.loc[volumen_iex, "volume"] = np.nan   # se imputará desde Yahoo al combinar (sección 6)
df_diario = df_diario.drop(columns=["volume_yahoo"])

Volúmenes del CSV marcados como solo-IEX: 2023 de 6374
Primer y último día afectados: 2020-06-02 -> 2021-06-30


Hecho esto, `df_diario` queda como la versión limpia del CSV (precios ajustados, volumen solo donde es consolidado).

## 5. Reparar `high`/`low` incoherentes con el cierre

Con el cierre bien calculado, el 27,7% de filas con el cierre fuera de [low, high] **desaparece del todo** en el diario: venía de que `close` era el cierre del día anterior, no de un error en los extremos. Se deja la reparación como salvaguarda por si en el futuro se cambian los datos o la regla del cierre: expandir el rango (`high = max(high, close)`, `low = min(low, close)`) en vez de borrar filas, porque en ese caso el error lógico estaría en los extremos y el Montecarlo solo usa `close`.

In [24]:
def contar_inconsistencias(d):
    return int(((d["high"] < d["low"]) | (d["close"] > d["high"]) | (d["close"] < d["low"])).sum())

print("Inconsistencias antes de reparar:", contar_inconsistencias(df_diario.dropna(subset=["close"])),
      "de", df_diario["close"].notna().sum())
df_diario["high"] = df_diario[["high", "close"]].max(axis=1)
df_diario["low"] = df_diario[["low", "close"]].min(axis=1)
print("Inconsistencias tras reparar:", contar_inconsistencias(df_diario.dropna(subset=["close"])))

Inconsistencias antes de reparar: 0 de 9027
Inconsistencias tras reparar: 0


## 6. Combinar: CSV primero, Yahoo imputa

**Calendario:** se toman como días válidos los días de mercado que devuelve Yahoo en la ventana. El CSV trae 95 filas en festivos de la bolsa de EE. UU. (4 de julio, Navidad, Acción de Gracias, el funeral de G. H. W. Bush el 05/12/2018…), solo en las 5 acciones del tramo 2018-2020 que siguen en el universo (AAPL, CAT, GS, NKE, V): la bolsa estaba cerrada y esas filas repiten el precio anterior. Se eliminan porque meterían retornos 0 que no existen y bajarían σ artificialmente.

**Prioridad, fila a fila:**
1. Si el CSV tiene cierre ese día → la fila es del CSV. Si le falta algún valor suelto (el volumen del tramo 2018-2020 viene vacío, y el solo-IEX se ha marcado como no válido en la sección 4b), ese valor se imputa desde Yahoo.
2. Si el CSV no tiene el día o no tiene cierre → la fila entera viene de Yahoo.

Se guardan dos columnas de trazabilidad: `fuente` (de dónde sale el precio) y `volume_imputado`.

In [25]:
festivos_csv = df_diario.merge(df_yahoo[["symbol", "date"]], on=["symbol", "date"], how="left", indicator=True)
festivos_csv = festivos_csv[festivos_csv["_merge"] == "left_only"]
print("Filas del CSV en días sin mercado (se eliminan):", len(festivos_csv))
print(festivos_csv["date"].dt.strftime("%Y-%m-%d %a").value_counts().head(8))

Filas del CSV en días sin mercado (se eliminan): 95
date
2018-07-04 Wed    5
2018-09-03 Mon    5
2018-11-22 Thu    5
2018-12-05 Wed    5
2018-12-25 Tue    5
2019-01-01 Tue    5
2019-01-21 Mon    5
2019-02-18 Mon    5
Name: count, dtype: int64


In [26]:
columnas = ["open", "high", "low", "close", "volume"]

base = df_yahoo[["symbol", "date"]]   # calendario: días de mercado según Yahoo
csv_valido = df_diario.dropna(subset=["close"])
combinado = (
    base
    .merge(csv_valido[["symbol", "date"] + columnas], on=["symbol", "date"], how="left")
    .merge(df_yahoo[["symbol", "date"] + columnas], on=["symbol", "date"], how="left", suffixes=("", "_yahoo"))
)

hay_csv = combinado["close"].notna()
combinado["fuente"] = np.where(hay_csv, "csv", "yahoo")
combinado["volume_imputado"] = combinado["volume"].isna() & combinado["volume_yahoo"].notna()

# Fila del CSV -> solo se imputan los valores sueltos que falten; sin CSV -> fila entera de Yahoo
for col in columnas:
    combinado[col] = combinado[col].fillna(combinado[f"{col}_yahoo"])

df_final = (
    combinado[["symbol", "date"] + columnas + ["fuente", "volume_imputado"]]
    .sort_values(["symbol", "date"])
    .reset_index(drop=True)
)
print(df_final.shape)
print("Nulos por columna:"); print(df_final[columnas].isna().sum())

(75420, 9)
Nulos por columna:
open      0
high      0
low       0
close     0
volume    0
dtype: int64


In [27]:
# Procedencia de cada acción: días del CSV vs días imputados desde Yahoo
procedencia = (
    df_final.groupby("symbol")
    .agg(
        desde=("date", "min"), hasta=("date", "max"), dias=("date", "size"),
        dias_csv=("fuente", lambda s: (s == "csv").sum()),
    )
)
procedencia["pct_csv"] = (procedencia["dias_csv"] / procedencia["dias"] * 100).round(1)
procedencia

,desde,hasta,dias,dias_csv,pct_csv
symbol,,,,,
AAPL,2016-01-04,2025-12-31,2514,768,30.5
AMGN,2016-01-04,2025-12-31,2514,270,10.7
AMZN,2016-01-04,2025-12-31,2514,0,0.0
AXP,2016-01-04,2025-12-31,2514,270,10.7
BA,2016-01-04,2025-12-31,2514,269,10.7
CAT,2016-01-04,2025-12-31,2514,765,30.4
CRM,2016-01-04,2025-12-31,2514,269,10.7
CSCO,2016-01-04,2025-12-31,2514,269,10.7
CVX,2016-01-04,2025-12-31,2514,270,10.7


In [28]:
# Comprobación de continuidad en las fronteras CSV <-> Yahoo: si el ajuste de splits
# estuviera mal, aparecerían retornos enormes justo en el cambio de fuente
df_final["log_ret"] = np.log(df_final["close"]).groupby(df_final["symbol"]).diff()
cambio_fuente = df_final["fuente"] != df_final.groupby("symbol")["fuente"].shift()
fronteras = df_final[cambio_fuente & df_final["log_ret"].notna()]
print(f"Fronteras CSV<->Yahoo: {len(fronteras)}")
print(f"|retorno| máximo en una frontera: {fronteras['log_ret'].abs().max():.4f}")
print(f"|retorno| máximo en toda la serie: {df_final['log_ret'].abs().max():.4f}")
df_final.loc[df_final["log_ret"].abs().nlargest(5).index, ["symbol", "date", "close", "fuente", "log_ret"]]

Fronteras CSV<->Yahoo: 226
|retorno| máximo en una frontera: 0.1800


|retorno| máximo en toda la serie: 0.2724


,symbol,date,close,fuente,log_ret
11112,BA,2020-03-16,129.610001,yahoo,-0.272444
55526,NVDA,2016-11-11,2.199250,yahoo,0.260876
67700,UNH,2025-04-17,454.109985,yahoo,-0.253341
21170,CVX,2020-03-18,55.049999,yahoo,-0.250062
16254,CRM,2020-08-26,273.410000,csv,0.235231


Si el ajuste de splits fallara, en la frontera de AAPL (31/08/2020) aparecería un −1,39 (ln 1/4). No hay nada así. El mayor retorno en una frontera (BA, 11/06/2020, ≈ −0,18) es real: ese día el Dow Jones cayó ~7% y BA venía de una semana de subidas y bajadas de más del 10% diario. Los retornos más grandes de toda la serie también son movimientos reales: BA y CVX en marzo de 2020 (COVID), NVDA 11/11/2016 (resultados, +30%), UNH 17/04/2025 (resultados) y CRM 26/08/2020 (resultados, el día del ejemplo de la sección 1). `log_ret` solo sirve para esta comprobación y no se guarda.

## 7. Universo final: el Dow Jones de 2025

El CSV trae la composición del índice de 2018-2021. Entre esa fecha y 2025 el Dow Jones ha cambiado cuatro valores:

| Fecha | Entra | Sale | Motivo |
|---|---|---|---|
| 26/02/2024 | AMZN (Amazon) | WBA (Walgreens) | WBA no está en el CSV ni en Yahoo (dejó de cotizar el 28/08/2025, tras la compra de Sycamore Partners) |
| 08/11/2024 | NVDA (Nvidia) | INTC (Intel) | |
| 08/11/2024 | SHW (Sherwin-Williams) | DOW (Dow Inc.) | DOW solo cotiza desde el 20/03/2019 |

**Qué se hace:**
- **INTC y DOW se descartan** aunque estén en el CSV: no forman parte del índice de 2025. De paso desaparece el problema de DOW (no existía antes de 2019) y el de WBA (sin datos en ninguna fuente).
- **Las 6 acciones del índice de 2025 que no están en el CSV** (AMZN, HON, NVDA, SHW, VZ, WMT) vienen **enteras de Yahoo** (`fuente = "yahoo"` en todas sus filas). Todas cotizaban ya en 2016 (aunque AMZN, NVDA y SHW todavía fuera del índice), así que cubren la ventana completa: por el criterio 4 se cogen igual.
- Como vienen solo de Yahoo, sus splits (NVDA 4:1 en 2021 y 10:1 en 2024, AMZN 20:1 en 2022, SHW 3:1 en 2021, WMT 3:1 en 2024) ya están ajustados en origen: no hay frontera CSV↔Yahoo donde se pueda colar un escalón.

Comprobamos que las 30 cubren la ventana completa:


In [29]:
dias_mercado = df_yahoo.loc[df_yahoo["symbol"] == "AAPL", "date"]   # calendario completo de referencia
cobertura = df_final.groupby("symbol")["date"].agg(["min", "max", "size"])
print("Acciones en df_final:", df_final["symbol"].nunique())
print("Acciones sin la ventana completa:", sorted(cobertura.index[cobertura["size"] < len(dias_mercado)]))
print("Solo Yahoo (no están en el CSV):", sorted(procedencia.index[procedencia["dias_csv"] == 0]))


Acciones en df_final: 30
Acciones sin la ventana completa: []
Solo Yahoo (no están en el CSV): ['AMZN', 'HON', 'NVDA', 'SHW', 'VZ', 'WMT']


## 8. Guardar

`Datos/Transformados/` no se sube al repositorio (regla del curso).
- `dowjones_csv_diario_limpio.csv`: solo el CSV, a frecuencia diaria, ajustado por splits y reparado. Sirve de referencia de lo que aporta el CSV.
- `dowjones_diario_2016_2025.csv`: **el dataset de trabajo** (CSV + imputación de Yahoo, 2016-2025), con `fuente` y `volume_imputado`.

`dowjones_yahoo_diario.csv` (la versión anterior, solo Yahoo 2016-2021) deja de generarse porque ya no se usa; si sigue en la carpeta, se puede borrar.

In [30]:
df_final = df_final.drop(columns=["log_ret"])
df_diario.to_csv("Datos/Transformados/dowjones_csv_diario_limpio.csv", index=False)
df_final.to_csv("Datos/Transformados/dowjones_diario_2016_2025.csv", index=False)
print("Guardado:", df_diario.shape, df_final.shape)

Guardado: (9114, 8) (75420, 9)


## Pendientes para la próxima sesión

1. **Días con discrepancia CSV vs Yahoo > 2%** (sección 4): se han dejado los del CSV por el criterio de prioridad. Confirmar con los profesores si se mantienen o si una discrepancia grande justifica tratar el dato del CSV como erróneo e imputarlo.
2. **Volumen del CSV desde el 02/03/2021** (sección 4b): es volumen solo de IEX (~3% del total) y se ha imputado desde Yahoo. Es una excepción al criterio de que el CSV manda: confirmarla.
3. **Dividendos:** el `Close` (el del CSV y el de Yahoo) no está ajustado por dividendos. Para la valoración con Montecarlo bajo medida neutral al riesgo lo normal es usar el precio sin dividendos y tratar el rendimiento por dividendo aparte (o ignorarlo si el enunciado no lo pide). Confirmarlo antes de calibrar.